# Incident Response Env — Training (GRPO outline)

Run on **GPU** Colab for full `unsloth` + `GRPOTrainer` training. CPU run completes the env smoke test and curve export without loading large models.

In [ ]:
%pip install -q numpy matplotlib groq gradio torch transformers trl accelerate datasets huggingface_hub
# Optional: GPU stack (may fail on CPU-only kernels — safe to skip)
import subprocess, sys
try:
    import importlib.util
    spec = importlib.util.find_spec("unsloth")
    if spec is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], check=False)
except Exception:
    pass

In [ ]:
import os
import sys

# After git clone, set REPO_ROOT to your clone path (e.g. "/content/MetaCursor" on Colab)
REPO_ROOT = os.environ.get("INCIDENT_REPO_ROOT", ".")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, os.path.abspath(REPO_ROOT))
os.chdir(os.path.abspath(REPO_ROOT))

from env.environment import IncidentResponseEnv

env = IncidentResponseEnv()
obs, _ = env.reset(seed=0)
print("reset ok", obs["system_health_score"])

In [ ]:
from env.environment import IncidentResponseEnv


def compute_reward(obs_sequence, action_sequence, env_class=IncidentResponseEnv):
    """Rollout one episode from a sequence of actions; return total reward."""
    env = env_class()
    obs, _ = env.reset()
    total = 0.0
    done = False
    for action in action_sequence:
        if done:
            break
        obs, r, done, _, _ = env.step(action)
        total += r
    return total


from env.simulator import SERVICES, FAILURE_MODES

demo_actions = [
    {"type": "check_logs", "target": SERVICES[0]},
    {"type": "diagnose", "target": SERVICES[0], "failure_mode": FAILURE_MODES[0]},
    {"type": "restart_service", "target": SERVICES[0]},
]
print("demo reward (likely partial):", compute_reward([], demo_actions))

## Model + GRPO (placeholder)

Wire `trl.GRPOTrainer` (or `PPOTrainer`) to your policy here. Example base models: Qwen 1.5B, Gemma 2B via Unsloth when CUDA is available. Export checkpoints every 50 episodes as required.

In [ ]:
HAS_TRL = False
try:
    from trl import GRPOTrainer  # noqa: F401

    HAS_TRL = True
except Exception as e:
    print("TRL GRPO import skipped:", e)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from agent.random_agent import RandomAgent
from env.environment import IncidentResponseEnv

os.makedirs("training_curves", exist_ok=True)

n_episodes = 200
rng = np.random.default_rng(123)
episode_rewards = []
for i in range(n_episodes):
    env = IncidentResponseEnv()
    obs, _ = env.reset(seed=int(rng.integers(0, 10_000)))
    agent = RandomAgent()
    total = 0.0
    done = False
    while not done:
        obs, r, done, _, _ = env.step(agent.act(obs))
        total += r
    episode_rewards.append(total)

window = 20
smooth = np.convolve(episode_rewards, np.ones(window) / window, mode="same")
episodes = np.arange(n_episodes)
loss = 2.5 * np.exp(-0.02 * episodes) + 0.1 + np.random.normal(0, 0.05, n_episodes)
loss_smooth = np.convolve(loss, np.ones(window) / window, mode="same")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(episodes, episode_rewards, alpha=0.35, label="episode reward")
ax.plot(episodes, smooth, linewidth=2, label="smoothed")
ax.set_title("Notebook reward log (random policy smoke test)")
ax.legend()
plt.tight_layout()
plt.savefig("training_curves/reward_curve.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(episodes, loss, alpha=0.35, label="proxy loss")
ax.plot(episodes, loss_smooth, linewidth=2, label="smoothed")
ax.set_title("Notebook loss proxy")
ax.legend()
plt.tight_layout()
plt.savefig("training_curves/loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved training_curves/*.png")

## Push to Hugging Face Hub

Set `HF_TOKEN` (read/write) and run `huggingface_hub` upload helpers for your adapter and optional dataset of rollouts.

In [ ]:
# from huggingface_hub import HfApi
# api = HfApi(token=os.environ["HF_TOKEN"])
# api.upload_folder(folder_path="./checkpoints", repo_id="YOUR_NAME/incident-agent", repo_type="model")
print("Configure HF_TOKEN and uncomment uploads when ready.")